# Fantasy Football Analyzer -- League Walkthrough

This notebook walks through the current `fantasy_analyzer` package against
**your own** Sleeper data: discovering your leagues, building a normalized
`LeagueSnapshot`, computing standings, normalizing the full season of
matchups, and running the Epic 5 (head-to-head) and Epic 6 (advanced league
analytics) query services described in `AGENTS.md`.

It makes **live calls to the Sleeper API** -- nothing here uses test
fixtures. Re-run cells freely; `SleeperClient` is read-only and Sleeper's
API is unauthenticated for public data.

**Player-level analytics** (Epic 7 / FFA-060+) are intentionally *not*
covered beyond basic roster player metadata: only the provider *interface*
(FFA-060) exists in the package so far. The `nflverse` stat provider
(FFA-061), the Sleeper<->nflverse ID crosswalk (FFA-062), and the
league-scoring engine (FFA-063) are still backlog items, so there is no
player-week fact table to analyze yet. The last section shows what *is*
available today.

## Prerequisites

This notebook needs a Jupyter kernel running the project's virtual
environment. From the repo root:

```bash
.venv/bin/pip install -e ".[dev]" ipykernel
.venv/bin/python -m ipykernel install --user --name fantasy-analyzer
```

Then select the `fantasy-analyzer` kernel for this notebook.


In [ ]:
!source .venv/bin/activate
%pip install -e ".[dev]"

In [25]:
from __future__ import annotations

import pandas as pd

from fantasy_analyzer.analytics.league_analytics import build_league_analytics
from fantasy_analyzer.analytics.matchup_history import build_matchup_history
from fantasy_analyzer.analytics.summary import build_league_summary
from fantasy_analyzer.league.season import derive_season_boundaries
from fantasy_analyzer.league.snapshot import load_league_snapshot
from fantasy_analyzer.matchups.loader import load_season_matchups
from fantasy_analyzer.matchups.outcomes import derive_season_outcomes
from fantasy_analyzer.matchups.pairing import pair_season_matchups
from fantasy_analyzer.matchups.reconciliation import reconcile_matchups_to_standings
from fantasy_analyzer.matchups.season_matchups import build_season_matchup_df
from fantasy_analyzer.sleeper.client import SleeperClient

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# Per AGENTS.md: the primary development user, but code must not hard-code
# this into library logic -- it is only a notebook-local default. Change it
# to analyze a different Sleeper user.
SLEEPER_USERNAME = "schneidbaby"
SEASON = 2025

client = SleeperClient()


## 1. Discover your leagues (FFA-003)

Resolve your username to Sleeper's immutable `user_id`, then list every
league you're in for `SEASON`.


In [26]:
user = client.get_user(SLEEPER_USERNAME)
user_id = user["user_id"]

leagues = client.get_leagues(user_id=user_id, season=SEASON)
leagues_df = pd.DataFrame(
    [{"league_id": lg["league_id"], "name": lg["name"], "status": lg["status"]} for lg in leagues]
)
leagues_df


,league_id,name,status
0,1283213576168837120,BBC weird league,complete
1,1268699066192101376,NWC Guillotine League,complete
2,1262800342051999744,Just Here For The Zipline,complete
3,1260307567133859840,N(ew Wave) F(riends) L(eague),complete
4,1257477810625196032,"NWC FFL, est. 2011",complete


Pick a `league_id` from the table above and set it below. Defaults to the
first league returned if you don't change it.


In [27]:
LEAGUE_ID = leagues[3]["league_id"]
# LEAGUE_ID = leagues_df[leagues_df["name"] == 'N(ew Wave) F(riends) L(eague']["league_id"]

# 2025 fantasy season length. Sleeper doesn't expose "total weeks" directly
# (see league/season.py) -- this must be supplied per-season, not hardcoded
# into library logic. 18 matches the CLI's README example; adjust if your
# league's playoffs end on a different final week.
TOTAL_WEEKS = 18

LEAGUE_ID


'1260307567133859840'

## 2. Build a `LeagueSnapshot` (FFA-013)

One call fetches league metadata, users, rosters, and the (locally cached)
player catalog, and normalizes them into the common input downstream
analytics expect -- no raw endpoint joins from here on.


In [28]:
snapshot = load_league_snapshot(client, LEAGUE_ID)

print(f"{snapshot.league.name} -- season {snapshot.league.season} ({snapshot.league.status})")
print(f"{snapshot.league.total_rosters} teams, playoffs start week {snapshot.league.playoff_week_start}")


N(ew Wave) F(riends) L(eague) -- season 2025 (complete)
10 teams, playoffs start week 15


In [29]:
snapshot.teams_df


,roster_id,owner_id,display_name,team_name
0,1,1124895994132406272,callahanmancan,Riley's Little Ladds
1,2,1124901188417331200,hankleeman,Riley Doesn’t Run
2,3,865610532865060864,schneidbaby,Bar Rat Boyfriends
3,4,1127727340412416000,danerunning,Larry’s Call Girl
4,5,1132821020680835072,jsnesbit,Fumble for a Tumble
5,6,1257880184422404096,TaylorDucote,"Live, Laugh, Jihad"
6,7,599792499258613760,Deschi,Desch n Dee
7,8,1128875516674605056,sleepymegan,Larry’s Caretaker
8,9,733854730526269440,Sticks24,Sanka ya Dead???
9,10,1127725968589746176,ayyeeohh,10th st on top


In [30]:
snapshot.rosters_df


,roster_id,owner_id,wins,losses,ties,fpts,fpts_against,players,starters
0,1,1124895994132406272,11,3,0,1875.48,1673.96,"[11533, 11579, 11620, 11631, 11635, 12527, 221...","[3257, 12527, 6790, 7569, 2216, 8130, 5045, 11..."
1,2,1124901188417331200,6,8,0,1935.72,1906.36,"[10229, 10236, 10859, 11564, 1945, 4199, 4943,...","[11564, 6813, 9221, 8126, 7525, 10236, 5947, 8..."
2,3,865610532865060864,4,10,0,1789.56,1963.54,"[11539, 11584, 12489, 12490, 12514, 12517, 213...","[6804, 12489, 11584, 12514, 7564, 12517, 8148,..."
3,4,1127727340412416000,10,4,0,1996.38,1729.64,"[11560, 11588, 12545, 4034, 4217, 4983, 5022, ...","[12545, 4034, 5967, 8137, 5872, 5022, 9226, 49..."
4,5,1132821020680835072,8,6,0,1729.74,1696.38,"[10213, 12711, 4984, 5012, 5859, 5870, 6130, 6...","[4984, 8132, 8150, 8134, 7547, 5012, 5859, 814..."
5,6,1257880184422404096,8,6,0,1826.68,1797.12,"[11586, 11628, 11786, 12508, 12529, 2449, 4381...","[6770, 4866, 12529, 9488, 2449, 7553, 4381, 11..."
6,7,599792499258613760,9,5,0,1860.48,1744.90,"[10222, 11589, 11624, 11655, 12506, 1479, 2747...","[3294, 8138, 9224, 6786, 10222, 12506, 5850, 1..."
7,8,1128875516674605056,4,10,0,1736.14,1924.74,"[11638, 12526, 1466, 17, 4035, 4892, 6794, 700...","[4892, 8155, 7588, 11638, 6794, 1466, 12526, 7..."
8,9,733854730526269440,7,7,0,1728.70,1768.42,"[12501, 12507, 12518, 12533, 2020, 4881, 5927,...","[4881, 9509, 12533, 5927, 9997, 8110, 12518, 7..."
9,10,1127725968589746176,3,11,0,1634.88,1908.70,"[10219, 10232, 11647, 12493, 1339, 3198, 4033,...","[6904, 8151, 3198, 9493, 5846, 12493, 10232, 7..."


In [31]:
# A quick look at the scoring settings and roster slots driving this league.
print(snapshot.roster_positions)
pd.Series(snapshot.scoring_settings).sort_index()


['QB', 'RB', 'RB', 'WR', 'WR', 'TE', 'FLEX', 'FLEX', 'K', 'DEF', 'BN', 'BN', 'BN', 'BN', 'BN']


blk_kick            2.00
def_st_ff           1.00
def_st_fum_rec      1.00
def_st_td           6.00
def_td              6.00
ff                  1.00
fgm_0_19            3.00
fgm_20_29           3.00
fgm_30_39           3.00
fgm_40_49           4.00
fgm_50p             5.00
fgmiss             -1.00
fum                 0.00
fum_lost           -2.00
fum_rec             2.00
fum_rec_td          6.00
int                 2.00
pass_2pt            2.00
pass_int           -1.00
pass_td             4.00
pass_yd             0.04
pts_allow_0        10.00
pts_allow_14_20     1.00
pts_allow_1_6       7.00
pts_allow_21_27     0.00
pts_allow_28_34    -1.00
pts_allow_35p      -4.00
pts_allow_7_13      4.00
rec                 1.00
rec_2pt             2.00
rec_td              6.00
rec_yd              0.10
rush_2pt            2.00
rush_td             6.00
rush_yd             0.10
sack                1.00
safe                2.00
st_ff               1.00
st_fum_rec          1.00
st_td               6.00


## 3. League summary: standings and scoring (FFA-020/021/023)

`LeagueSummary` wraps the snapshot with `total_weeks` and exposes
`standings()`, `scoring_summary()`, and the combined `league_summary()`
view -- the same service the `fantasy-analyzer summary` CLI command uses.


In [32]:
summary = build_league_summary(snapshot, total_weeks=TOTAL_WEEKS)

summary.standings()


,roster_id,owner_id,display_name,team_name,wins,losses,ties,win_pct,points_for,points_against,point_diff,rank
0,1,1124895994132406272,callahanmancan,Riley's Little Ladds,11,3,0,0.785714,1875.48,1673.96,201.52,1
1,4,1127727340412416000,danerunning,Larry’s Call Girl,10,4,0,0.714286,1996.38,1729.64,266.74,2
2,7,599792499258613760,Deschi,Desch n Dee,9,5,0,0.642857,1860.48,1744.90,115.58,3
3,6,1257880184422404096,TaylorDucote,"Live, Laugh, Jihad",8,6,0,0.571429,1826.68,1797.12,29.56,4
4,5,1132821020680835072,jsnesbit,Fumble for a Tumble,8,6,0,0.571429,1729.74,1696.38,33.36,5
5,9,733854730526269440,Sticks24,Sanka ya Dead???,7,7,0,0.500000,1728.70,1768.42,-39.72,6
6,2,1124901188417331200,hankleeman,Riley Doesn’t Run,6,8,0,0.428571,1935.72,1906.36,29.36,7
7,3,865610532865060864,schneidbaby,Bar Rat Boyfriends,4,10,0,0.285714,1789.56,1963.54,-173.98,8
8,8,1128875516674605056,sleepymegan,Larry’s Caretaker,4,10,0,0.285714,1736.14,1924.74,-188.60,9
9,10,1127725968589746176,ayyeeohh,10th st on top,3,11,0,0.214286,1634.88,1908.70,-273.82,10


In [33]:
summary.scoring_summary()


,roster_id,owner_id,display_name,team_name,points_per_game,points_against_per_game,scoring_rank,avg_margin
0,4,1127727340412416000,danerunning,Larry’s Call Girl,142.598571,123.545714,1,19.052857
1,2,1124901188417331200,hankleeman,Riley Doesn’t Run,138.265714,136.168571,2,2.097143
2,1,1124895994132406272,callahanmancan,Riley's Little Ladds,133.962857,119.568571,3,14.394286
3,7,599792499258613760,Deschi,Desch n Dee,132.891429,124.635714,4,8.255714
4,6,1257880184422404096,TaylorDucote,"Live, Laugh, Jihad",130.477143,128.365714,5,2.111429
5,3,865610532865060864,schneidbaby,Bar Rat Boyfriends,127.825714,140.252857,6,-12.427143
6,8,1128875516674605056,sleepymegan,Larry’s Caretaker,124.010000,137.481429,7,-13.471429
7,5,1132821020680835072,jsnesbit,Fumble for a Tumble,123.552857,121.170000,8,2.382857
8,9,733854730526269440,Sticks24,Sanka ya Dead???,123.478571,126.315714,9,-2.837143
9,10,1127725968589746176,ayyeeohh,10th st on top,116.777143,136.335714,10,-19.558571


## 4. Normalize the full season of matchups (FFA-030 through FFA-033)

The matchup pipeline is four explicit stages -- load -> pair -> derive
outcomes -> build the canonical season table -- each independently testable
and each already covered by unit tests against fixtures. Here we run it
against your live season.


In [34]:
boundaries = derive_season_boundaries(snapshot.league, total_weeks=TOTAL_WEEKS)
print(f"regular season weeks: {boundaries.regular_season_weeks}")
print(f"playoff weeks: {boundaries.playoff_weeks}")

weeks = load_season_matchups(client, LEAGUE_ID, boundaries, season=snapshot.league.season)
pairings = pair_season_matchups(weeks)
outcomes = derive_season_outcomes(pairings)
season_matchup_df = build_season_matchup_df(outcomes, snapshot.teams_df)

season_matchup_df.head(10)


regular season weeks: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
playoff weeks: [15, 16, 17, 18]


,season,week,is_playoff,matchup_id,roster_1_id,roster_2_id,owner_1,owner_2,points_1,points_2,winner,loser,is_tie,margin,point_differential
0,2025,1,False,3.0,1,8.0,callahanmancan,sleepymegan,108.42,153.98,8.0,1.0,False,45.56,-45.56
1,2025,1,False,1.0,2,5.0,hankleeman,jsnesbit,85.62,114.96,5.0,2.0,False,29.34,-29.34
2,2025,1,False,5.0,3,9.0,schneidbaby,Sticks24,127.42,140.06,9.0,3.0,False,12.64,-12.64
3,2025,1,False,4.0,4,6.0,danerunning,TaylorDucote,119.72,132.52,6.0,4.0,False,12.80,-12.80
4,2025,1,False,2.0,7,10.0,Deschi,ayyeeohh,109.20,139.68,10.0,7.0,False,30.48,-30.48
5,2025,2,False,4.0,1,9.0,callahanmancan,Sticks24,100.38,124.90,9.0,1.0,False,24.52,-24.52
6,2025,2,False,2.0,2,7.0,hankleeman,Deschi,114.84,149.56,7.0,2.0,False,34.72,-34.72
7,2025,2,False,5.0,3,4.0,schneidbaby,danerunning,144.00,156.38,4.0,3.0,False,12.38,-12.38
8,2025,2,False,1.0,5,10.0,jsnesbit,ayyeeohh,147.82,116.44,5.0,10.0,False,31.38,31.38
9,2025,2,False,3.0,6,8.0,TaylorDucote,sleepymegan,135.50,121.30,6.0,8.0,False,14.20,14.20


### Sanity check: reconcile derived totals against Sleeper's standings (FFA-034)

Independently re-derives each roster's wins/losses/ties/points from the
matchup-level data and compares it against Sleeper's own cumulative
counters. Any row with `reconciled == False` below is worth investigating
(e.g. a week Sleeper hasn't finished scoring yet).


In [35]:
reconciliation_df = reconcile_matchups_to_standings(season_matchup_df, summary.standings())
mismatches = reconciliation_df.loc[~reconciliation_df["reconciled"]]

if mismatches.empty:
    print("All rosters reconcile within tolerance.")
else:
    print(f"{len(mismatches)} roster(s) do not reconcile -- likely an in-progress week:")
mismatches


10 roster(s) do not reconcile -- likely an in-progress week:


,roster_id,derived_wins,sleeper_wins,wins_diff,wins_match,derived_losses,sleeper_losses,losses_diff,losses_match,derived_ties,sleeper_ties,ties_diff,ties_match,derived_points_for,sleeper_points_for,points_for_diff,points_for_match,derived_points_against,sleeper_points_against,points_against_diff,points_against_match,reconciled
0,1,12,11,1.0,False,4,3,1,False,0,0,0,True,2347.16,1875.48,471.68,False,1908.66,1673.96,234.70,False,False
1,2,7,6,1.0,False,9,8,1,False,0,0,0,True,2389.90,1935.72,454.18,False,2148.96,1906.36,242.60,False,False
2,3,5,4,1.0,False,11,10,1,False,0,0,0,True,2237.66,1789.56,448.10,False,2247.32,1963.54,283.78,False,False
3,4,12,10,2.0,False,4,4,0,True,0,0,0,True,2501.34,1996.38,504.96,False,1978.80,1729.64,249.16,False,False
4,5,8,8,0.0,True,8,6,2,False,0,0,0,True,2124.14,1729.74,394.40,False,1950.96,1696.38,254.58,False,False
5,6,10,8,2.0,False,7,6,1,False,0,0,0,True,2301.28,1826.68,474.60,False,2167.78,1797.12,370.66,False,False
6,7,10,9,1.0,False,7,5,2,False,0,0,0,True,2364.18,1860.48,503.70,False,2154.60,1744.90,409.70,False,False
7,8,4,4,0.0,True,12,10,2,False,0,0,0,True,2098.88,1736.14,362.74,False,2220.96,1924.74,296.22,False,False
8,9,8,7,1.0,False,8,7,1,False,0,0,0,True,2181.36,1728.70,452.66,False,2017.18,1768.42,248.76,False,False
9,10,5,3,2.0,False,11,11,0,True,0,0,0,True,2195.48,1634.88,560.60,False,2097.88,1908.70,189.18,False,False


## 5. Head-to-head history (Epic 5: FFA-040 through FFA-043)

`MatchupHistory` composes head-to-head records, the head-to-head matrix,
and rivalry/margin stats. Lookups are keyed by `roster_id` (not owner name
-- see the module docstring for why), so first find your own roster.


In [36]:
history = build_matchup_history(season_matchup_df, snapshot.teams_df)

my_roster_id = int(snapshot.teams_df.loc[snapshot.teams_df["owner_id"] == user_id, "roster_id"].iloc[0])
my_roster_id


3

In [37]:
# The full owner-by-owner "W-L" grid.
history.formatted_head_to_head_matrix()


opponent_owner,callahanmancan,hankleeman,schneidbaby,danerunning,jsnesbit,TaylorDucote,Deschi,sleepymegan,Sticks24,ayyeeohh
owner,,,,,,,,,,
callahanmancan,—,2-0,2-0,1-0,2-0,0-2,2-0,1-1,1-1,1-0
hankleeman,0-2,—,1-0,0-1,1-1,0-1,0-2,2-1,1-0,2-1
schneidbaby,0-2,0-1,—,0-2,0-1,1-1,0-2,2-0,1-1,1-1
danerunning,0-1,1-0,2-0,—,1-0,2-1,2-0,2-0,1-1,1-1
jsnesbit,0-2,1-1,1-0,0-1,—,1-1,1-1,1-1,1-1,2-0
TaylorDucote,2-0,1-0,1-1,1-2,1-1,—,0-1,2-0,1-1,1-1
Deschi,0-2,2-0,2-0,0-2,1-1,1-0,—,1-0,2-1,1-1
sleepymegan,1-1,1-2,0-2,0-2,1-1,0-2,0-1,—,0-1,1-0
Sticks24,1-1,0-1,1-1,1-1,1-1,1-1,1-2,1-0,—,1-0


In [38]:
# Your record against every opponent you've played this season.
for opponent_roster_id in snapshot.teams_df["roster_id"]:
    if opponent_roster_id == my_roster_id:
        continue
    result = history.head_to_head(my_roster_id, opponent_roster_id)
    if result is None:
        continue
    print(
        f"vs {result.opponent_owner:20s} {result.wins}-{result.losses}-{result.ties}"
        f"  avg {result.avg_points:.1f} - {result.avg_opponent_points:.1f}"
    )


vs callahanmancan       0-2-0  avg 106.2 - 138.5
vs hankleeman           0-1-0  avg 157.2 - 161.8
vs danerunning          0-2-0  avg 113.8 - 169.3
vs jsnesbit             0-1-0  avg 112.0 - 120.5
vs TaylorDucote         1-1-0  avg 144.4 - 155.0
vs Deschi               0-2-0  avg 132.4 - 150.4
vs sleepymegan          2-0-0  avg 150.6 - 117.6
vs Sticks24             1-1-0  avg 133.0 - 126.2
vs ayyeeohh             1-1-0  avg 103.0 - 125.6


## 6. Advanced league analytics (Epic 6: FFA-050 through FFA-057)

`LeagueAnalytics` composes all-play standings, expected wins / schedule
luck, consistency metrics, strength of schedule, the documented power
ranking model, and the normalized playoff bracket / final placements. Every
metric here has a written definition in its own module -- see the
docstrings in `analytics/` for the exact formulas and edge-case rules this
notebook inherits unchanged.


In [39]:
# Playoff bracket data is optional -- omit it and playoff_brackets()/
# final_placements() simply return empty frames (e.g. before the postseason
# has started).
try:
    winners_bracket_raw = client.get_winners_bracket(LEAGUE_ID)
    losers_bracket_raw = client.get_losers_bracket(LEAGUE_ID)
except Exception:
    winners_bracket_raw, losers_bracket_raw = None, None

analytics = build_league_analytics(
    season_matchup_df,
    snapshot.teams_df,
    winners_bracket_raw=winners_bracket_raw,
    losers_bracket_raw=losers_bracket_raw,
)


### All-play standings (FFA-051)

How each team would have fared against *every* other team each week, not just its actual opponent.

In [40]:
analytics.all_play()

,roster_id,owner,weeks_played,all_play_wins,all_play_losses,all_play_ties,all_play_games,all_play_win_pct,all_play_rank
0,4,danerunning,18,110,52,0,162,0.679012,1
1,7,Deschi,18,93,69,0,162,0.574074,2
2,2,hankleeman,18,90,72,0,162,0.555556,3
3,1,callahanmancan,18,88,74,0,162,0.543210,4
4,6,TaylorDucote,18,81,81,0,162,0.500000,5
5,3,schneidbaby,18,77,85,0,162,0.475309,6
6,10,ayyeeohh,18,76,86,0,162,0.469136,7
7,9,Sticks24,18,72,90,0,162,0.444444,8
8,8,sleepymegan,18,62,100,0,162,0.382716,9
9,5,jsnesbit,18,61,101,0,162,0.376543,10


### Expected wins and schedule luck (FFA-052)

`schedule_luck` is signed: positive means the actual record outperformed the all-play (schedule-independent) rate.

In [41]:
analytics.schedule_luck().sort_values('schedule_luck', ascending=False)

,roster_id,owner,games_played,wins,losses,ties,win_pct,all_play_games,all_play_win_pct,all_play_rank,expected_wins,expected_losses,schedule_luck,schedule_luck_rank
0,1,callahanmancan,16,12,4,0,0.750000,162,0.543210,4,8.691358,7.308642,3.308642,1
1,5,jsnesbit,16,8,8,0,0.500000,162,0.376543,10,6.024691,9.975309,1.975309,2
2,6,TaylorDucote,17,10,7,0,0.588235,162,0.500000,5,8.500000,8.500000,1.500000,3
3,4,danerunning,16,12,4,0,0.750000,162,0.679012,1,10.864198,5.135802,1.135802,4
4,9,Sticks24,16,8,8,0,0.500000,162,0.444444,8,7.111111,8.888889,0.888889,5
5,7,Deschi,17,10,7,0,0.588235,162,0.574074,2,9.759259,7.240741,0.240741,6
6,2,hankleeman,16,7,9,0,0.437500,162,0.555556,3,8.888889,7.111111,-1.888889,7
7,8,sleepymegan,16,4,12,0,0.250000,162,0.382716,9,6.123457,9.876543,-2.123457,8
8,10,ayyeeohh,16,5,11,0,0.312500,162,0.469136,7,7.506173,8.493827,-2.506173,9
9,3,schneidbaby,16,5,11,0,0.312500,162,0.475309,6,7.604938,8.395062,-2.604938,10


### Consistency (FFA-053)

Weekly scoring spread per team: standard deviation, coefficient of variation, floor/ceiling, and boom/bust frequency.

In [42]:
analytics.consistency()

,roster_id,owner,weeks_played,mean_points,median_points,stdev_points,cv,scoring_floor,scoring_ceiling,boom_weeks,boom_pct,bust_weeks,bust_pct
0,1,callahanmancan,18,130.397778,126.33,24.281903,0.186214,73.22,168.76,4.0,0.222222,3.0,0.166667
1,2,hankleeman,18,132.772222,127.00,32.545972,0.245126,85.62,193.78,3.0,0.166667,4.0,0.222222
2,3,schneidbaby,18,124.314444,120.21,23.805317,0.191493,83.70,164.76,4.0,0.222222,3.0,0.166667
3,4,danerunning,18,138.963333,142.50,26.748232,0.192484,67.06,182.24,2.0,0.111111,3.0,0.166667
4,5,jsnesbit,18,118.007778,121.44,21.335591,0.180798,66.30,153.38,3.0,0.166667,2.0,0.111111
5,6,TaylorDucote,18,127.848889,126.10,22.348299,0.174802,89.00,169.96,3.0,0.166667,4.0,0.222222
6,7,Deschi,18,131.343333,135.43,27.925676,0.212616,68.50,178.34,2.0,0.111111,3.0,0.166667
7,8,sleepymegan,18,116.604444,115.40,28.315679,0.242835,61.12,165.46,4.0,0.222222,2.0,0.111111
8,9,Sticks24,18,121.186667,121.70,23.758836,0.196052,78.42,172.48,2.0,0.111111,3.0,0.166667
9,10,ayyeeohh,18,121.971111,121.37,24.951102,0.204566,83.00,168.00,4.0,0.222222,4.0,0.222222


### Strength of schedule (FFA-054)

Average opponent strength faced, by two measures -- see `strength_of_schedule.py` for the rank-1-is-toughest convention.

In [43]:
analytics.strength_of_schedule()

,roster_id,owner,opponent_weeks,unresolved_opponent_weeks,avg_opponent_win_pct,sos_rank,avg_opponent_all_play_win_pct,all_play_sos_rank
0,9,Sticks24,16,0,0.535386,1,0.517361,2
1,8,sleepymegan,16,0,0.532169,2,0.518904,1
2,3,schneidbaby,16,0,0.525965,3,0.507330,4
3,10,ayyeeohh,16,0,0.518153,4,0.515432,3
4,6,TaylorDucote,17,0,0.501514,5,0.502905,6
5,7,Deschi,17,0,0.497837,6,0.494916,7
6,5,jsnesbit,16,0,0.494715,7,0.505787,5
7,2,hankleeman,16,0,0.469669,8,0.477623,10
8,1,callahanmancan,16,0,0.463465,9,0.485340,8
9,4,danerunning,16,0,0.461167,10,0.479167,9


### Power rankings (FFA-056)

A transparent, documented composite of `win_pct`, `all_play_win_pct`, and
`mean_points` (each z-scored and weighted) -- deliberately **not** an
opaque score. See `power_rankings.py` for the exact weights and why
schedule luck and consistency are excluded from the composite.


In [44]:
analytics.power_rankings()

,roster_id,owner,games_played,win_pct,all_play_win_pct,mean_points,win_pct_z,all_play_win_pct_z,mean_points_z,power_score,power_rank
0,4,danerunning,16,0.750000,0.679012,138.963333,1.507976,2.051636,1.864202,1.851051,1
1,1,callahanmancan,16,0.750000,0.543210,130.397778,1.507976,0.495222,0.599149,0.819834,2
2,7,Deschi,17,0.588235,0.574074,131.343333,0.536513,0.848953,0.738798,0.733190,3
3,2,hankleeman,16,0.437500,0.555556,132.772222,-0.368715,0.636715,0.949832,0.397709,4
4,6,TaylorDucote,17,0.588235,0.500000,127.848889,0.536513,0.000000,0.222701,0.205494,5
5,9,Sticks24,16,0.500000,0.444444,121.186667,0.006624,-0.636715,-0.761247,-0.468620,6
6,3,schneidbaby,16,0.312500,0.475309,124.314444,-1.119391,-0.282984,-0.299303,-0.537170,7
7,10,ayyeeohh,16,0.312500,0.469136,121.971111,-1.119391,-0.353730,-0.645392,-0.641761,8
8,5,jsnesbit,16,0.500000,0.376543,118.007778,0.006624,-1.414921,-1.230740,-0.951621,9
9,8,sleepymegan,16,0.250000,0.382716,116.604444,-1.494729,-1.344175,-1.437999,-1.408106,10


### Playoff bracket and final placements (FFA-055)

Empty until your league's playoffs have started / been seeded.

In [45]:
analytics.playoff_brackets()

,bracket,round,match_id,roster_1_id,roster_2_id,roster_1_from,roster_2_from,winner_roster_id,loser_roster_id,winner_placement,loser_placement
0,winners,1,1,6,5,None,None,6,5,None,None
1,winners,1,2,9,7,None,None,7,9,None,None
2,winners,2,3,1,6,None,winner_of_match_1,6,1,None,None
3,winners,2,4,4,7,None,winner_of_match_2,4,7,None,None
4,winners,2,5,5,9,loser_of_match_1,loser_of_match_2,9,5,5,6
5,winners,3,6,6,4,winner_of_match_3,winner_of_match_4,4,6,1,2
6,winners,3,7,1,7,loser_of_match_3,loser_of_match_4,1,7,3,4
7,losers,1,1,10,2,None,None,2,10,None,None
8,losers,1,2,8,3,None,None,8,3,None,None
9,losers,2,3,2,8,winner_of_match_1,winner_of_match_2,8,2,1,2


In [46]:
analytics.final_placements()

,roster_id,owner,placement,bracket,match_id
0,4,danerunning,1,winners,6
1,8,sleepymegan,1,losers,3
2,2,hankleeman,2,losers,3
3,6,TaylorDucote,2,winners,6
4,1,callahanmancan,3,winners,7
5,3,schneidbaby,3,losers,4
6,7,Deschi,4,winners,7
7,10,ayyeeohh,4,losers,4
8,9,Sticks24,5,winners,5
9,5,jsnesbit,6,winners,5


## 7. Player data -- what's available today (Epic 7, in progress)

Only FFA-012 (roster player metadata) and FFA-060 (the provider *interface*
for weekly stats) exist so far. There is no nflverse ingestion, ID
crosswalk, league-scoring engine, or player-week fact table yet (FFA-061
through FFA-072 are all still `BACKLOG`), so there's no `player_value_df`
or `roster_efficiency_df` to show. What we *can* do today is resolve your
own roster's Sleeper player IDs into readable metadata via
`snapshot.players_df`.


In [47]:
my_roster = snapshot.rosters_df.loc[snapshot.rosters_df["roster_id"] == my_roster_id].iloc[0]
my_player_ids = set(my_roster["players"])

my_players_df = snapshot.players_df.loc[snapshot.players_df["player_id"].isin(my_player_ids)]
my_players_df.sort_values(["position", "full_name"])


,player_id,full_name,position,team
47,BAL,Baltimore Ravens,DEF,BAL
32,11539,Jake Bates,K,DET
42,6804,Jordan Love,QB,GB
40,421,Matthew Stafford,QB,LAR
35,12490,Bhayshul Tuten,RB,JAX
33,11584,Bucky Irving,RB,TB
41,5892,David Montgomery,RB,HOU
39,4098,Kareem Hunt,RB,NaN
34,12489,RJ Harvey,RB,DEN
46,9753,Zach Charbonnet,RB,SEA


## Summary

- **Sleeper client, league normalization, league summary, matchup
  normalization, head-to-head, and advanced league analytics (Epics 1-6,
  FFA-001 through FFA-057)** are implemented and exercised live above.
- **Player / position analytics (Epic 7, FFA-060 through FFA-072)** has
  only its provider interface (FFA-060) built. The next unlocking ticket is
  FFA-061 (nflverse weekly stat provider), per `AGENTS.md`'s dependency
  ordering -- once that lands, this notebook can grow a player-performance
  section the same way it grew the league-analytics section above.
